# MatterGen Colab Tutorial for Computational Chemistry Students

> **Goal:** Learn how to install MatterGen in Google Colab, understand the theory and architecture, run unconditional and property-conditioned generation, and visualize generated inorganic crystal structures.

This notebook is intentionally **verbose** and is designed as a teaching resource for students who are new to generative models in materials science.


## 0. How to use this notebook

- If you are in **Google Colab**, select **Runtime → Change runtime type → GPU**.
- Run cells in order from top to bottom.
- Some generation steps can take several minutes depending on GPU type.

### What you will learn
1. The methodological context of MatterGen (why diffusion for crystals).
2. The high-level architecture and training recipe.
3. Practical generation workflows (unconditional and conditioned).
4. How to inspect, parse, and visualize outputs.


## 1. Install prerequisites (Colab-friendly)

This section installs MatterGen and companion libraries for analysis/visualization.


In [ ]:
import sys
import subprocess


def run(cmd):
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

run(f"{sys.executable} -m pip install -U pip setuptools wheel")
run(f"{sys.executable} -m pip install -U git+https://github.com/microsoft/mattergen.git")
run(f"{sys.executable} -m pip install -U pymatgen ase py3Dmol pandas matplotlib seaborn tqdm")

print("\nInstallation complete.")


In [ ]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Methodology and theory (teaching notes)

### 2.1 Why generative modeling for computational chemistry?
Classical crystal discovery workflows (enumeration, substitution heuristics, random structure search, and high-throughput DFT) are powerful but expensive. We want a model that can:

- Learn structural priors from known inorganic crystals.
- Propose *novel* candidate structures with plausible chemistry.
- Be steerable toward target properties (e.g., band gap, magnetic density).

### 2.2 Why diffusion models?
MatterGen uses a diffusion-style approach where generation is treated as iterative denoising from noise to a crystal representation. Conceptually:

1. **Forward process:** progressively corrupts a training crystal with noise.
2. **Reverse process:** neural network learns to remove that noise step-by-step.
3. **Sampling:** start from noise at inference time and apply learned reverse dynamics.

Diffusion models are attractive because they are generally stable to train, scale well, and support guidance/conditioning strategies.

### 2.3 Crystal-specific representation challenge
Crystals are not simple fixed-length vectors: they involve

- Atomic species (discrete)
- Atomic positions (continuous, periodic)
- Lattice geometry (continuous, constrained)

MatterGen's architecture combines geometric deep learning ideas with diffusion modeling so these components can be denoised coherently.

### 2.4 Conditioning and classifier-free guidance intuition
For property-conditioned generation, MatterGen can be trained/fine-tuned with condition signals. During sampling, a guidance scale (diffusion guidance factor) can trade off:

- **Higher guidance:** better property adherence
- **Lower guidance:** more diversity/realism

This is analogous to classifier-free guidance in text-to-image diffusion, but adapted to materials property conditioning.

### 2.5 How training works (high-level)
At a high level, training includes:

1. Sample a crystal structure from a training dataset.
2. Corrupt it at a random timestep.
3. Predict denoising targets (for positions/lattice/species related terms).
4. Optimize a weighted diffusion loss over many timesteps.
5. For conditioned variants, include property embeddings and conditional dropout for guidance.

In practice, the repository contains both base and fine-tuning workflows as scripts/configs; this notebook focuses on **usage** and **scientific interpretation** rather than reproducing full large-scale pretraining.


## 3. MatterGen architecture map (code-oriented orientation)

When you later inspect the repository, these modules are useful landmarks:

- `mattergen/diffusion/`: diffusion mechanics, losses, modules.
- `mattergen/generator.py`: sampling/generation orchestration.
- `mattergen/property_embeddings.py`: handling conditioning/property embeddings.
- `mattergen/scripts/generate.py`: CLI entrypoint for generation.
- `mattergen/scripts/finetune.py`: fine-tuning workflow for conditional models.

As a student exercise, try opening these files side-by-side and mapping each concept in Section 2 to implementation components.


## 4. Unconditional generation with a pre-trained checkpoint

We first generate candidate structures with the base model (`mattergen_base`).


In [ ]:
import subprocess
from pathlib import Path

results_dir = Path("results_unconditional")
results_dir.mkdir(exist_ok=True)

cmd = [
    "mattergen-generate",
    str(results_dir),
    "--pretrained-name=mattergen_base",
    "--batch_size=8",
    "--num_batches=1",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nDone. Files in output directory:")
for p in sorted(results_dir.iterdir()):
    print(" -", p.name)


## 5. Property-conditioned generation

Now we condition on an example target property. You can replace values/models with other available checkpoints.

> Tip: Start with moderate guidance (e.g., 1.5–2.5), then scan values to study quality-vs-adherence tradeoffs.


In [ ]:
from pathlib import Path
import subprocess

cond_results = Path("results_conditioned")
cond_results.mkdir(exist_ok=True)

condition_dict = "{'dft_mag_density': 0.15}"

cmd = [
    "mattergen-generate",
    str(cond_results),
    "--pretrained-name=dft_mag_density",
    "--batch_size=8",
    "--num_batches=1",
    f"--properties_to_condition_on={condition_dict}",
    "--diffusion_guidance_factor=2.0",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nDone. Files in conditioned output directory:")
for p in sorted(cond_results.iterdir()):
    print(" -", p.name)


## 6. Parse generated outputs into Python objects

MatterGen writes structures as `.extxyz` and/or zipped CIF files. We'll load `.extxyz` with ASE and convert to Pymatgen for chemistry-aware analysis.


In [ ]:
from ase.io import read as ase_read
from pymatgen.io.ase import AseAtomsAdaptor
import pandas as pd

extxyz_path = "results_conditioned/generated_crystals.extxyz"
atoms_list = ase_read(extxyz_path, index=":")

structures = [AseAtomsAdaptor.get_structure(a) for a in atoms_list]

rows = []
for i, s in enumerate(structures):
    rows.append(
        {
            "sample_id": i,
            "formula": s.composition.reduced_formula,
            "n_sites": len(s),
            "volume": float(s.volume),
            "density": float(s.density),
        }
    )

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} structures")
df.head(10)


## 7. Quick statistical visualization

These plots help evaluate whether outputs look physically plausible and chemically diverse.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("talk")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["n_sites"], bins=15, kde=False, ax=axes[0])
axes[0].set_title("Distribution of atom counts per generated crystal")
axes[0].set_xlabel("Number of sites")

sns.histplot(df["volume"], bins=15, kde=False, ax=axes[1])
axes[1].set_title("Distribution of unit-cell volumes")
axes[1].set_xlabel("Volume (Å³)")

plt.tight_layout()
plt.show()


## 8. 3D visualization of a generated crystal

We'll render one generated structure using `py3Dmol` directly in notebook output.


In [ ]:
import py3Dmol
from pymatgen.io.cif import CifWriter
import tempfile

idx = 0
s = structures[idx]

with tempfile.NamedTemporaryFile(suffix=".cif", mode="w+", delete=False) as tmp:
    CifWriter(s).write_file(tmp.name)
    with open(tmp.name) as fh:
        cif_text = fh.read()

viewer = py3Dmol.view(width=700, height=500)
viewer.addModel(cif_text, "cif")
viewer.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.3}})
viewer.addUnitCell()
viewer.zoomTo()
viewer.show()


## 9. Suggested student exercises

1. **Guidance sweep:** Repeat conditioned generation with guidance factors `[0.0, 1.0, 2.0, 4.0]` and compare diversity/target adherence.
2. **Target sweep:** For a conditioned model, vary target values and inspect whether generated distributions shift in expected directions.
3. **Post-generation relaxation:** Use robust atomistic relaxation workflows before drawing thermodynamic conclusions.
4. **Novelty/uniqueness evaluation:** Use MatterGen evaluation tools to compute benchmark-style metrics.

These exercises are ideal for a computational chemistry classroom lab where students connect generative ML outputs with physical interpretation.


## 10. Troubleshooting (Colab)

- **Out of memory:** reduce `--batch_size`.
- **Slow runtime:** ensure GPU runtime is selected.
- **Package conflicts:** restart runtime and rerun setup cells.
- **Checkpoint download delays:** first call to a checkpoint may take time while files are fetched.
